In [6]:
# load standard libraries
import time # for runtime analysis
import numpy as np   # standard numerics library
import matplotlib.pyplot as plt   # for making plots
import matplotlib.ticker as ticker
from numpy import linalg as LA   # diagonalization and more
from scipy import sparse 
from scipy import linalg as sLA   # diagonalization and more
from scipy.sparse.linalg import eigsh
import Comp_Quant_Dynam as cqd  # custom quantum dynamics package

%matplotlib inline

# Lecture 6: ODE solvers

### Schrödinger Equation (SE)

**Goal:** Solve the time-dependent Schrödinger Equation (SE):

$$\dot{\vec{\psi}} = -i H \vec{\psi}, \quad \vec{\psi}(t=0) = \vec{\psi}_0$$

Formally, for a time-independent Hamiltonian $H$:

$$\vec{\psi}(t) = e^{-iHt} \vec{\psi}_0$$

### we used exact-diagonalization:

$$H = V D V^\dagger$$

* Where eigenvectors of $H$ are the columns of $V$
* Eigenvalues of $H$ are on the diagonal of $D$ 

$$\Rightarrow \vec{\psi}(t) = V e^{-iDt} V^\dagger \vec{\psi}_0$$

Where $e^{-iDt}$ is the diagonal matrix:

$$\begin{pmatrix}
e^{-iE_1 t} & & 0 \\
& \ddots & \\
0 & & e^{-iE_d t}
\end{pmatrix}$$

Diagonalization means finding the eigenvalues and eigenvectors.

So the purpose of exact diagonalization is not only to diagonalize the Hamiltonian \(H\), but also to use its eigenvalues and eigenvectors to compute the exact quantum time evolution of the system.

Diagonalization itself means finding eigenvalues and eigenvectors, but in quantum mechanics we use that diagonalization to compute the exact time evolution.

So the full purpose becomes:

1. diagonalize the Hamiltonian \(H\)
2. compute the exact solution of the time-dependent Schrödinger Equation

In [11]:
# Example  Hamiltonian H  Simple 2x2 Hamiltonian
H = np.array([[1, 0.2],[0.2, 2]], dtype=complex)
psi0 = np.array([1, 0], dtype=complex)
t = 1
# Exact diagonalization (H = V D V^) and Diagonal matrix D
eigenvalues, eigenvectors = np.linalg.eigh(H) 
D = np.diag(eigenvalues)

print("D=\n", D)
# psi(t) = V exp(-iDt) V^dagger psi0
# exp(-iDt) and psi(t)
expD = np.diag(np.exp(-1j * eigenvalues * t))
psi_t = V @ expD @ V.conj().T @ psi0
# Results
print("Eigenvalues=", eigenvalues)
print("Eigenvectors=\n", V)
print("psi(t)=", psi_t)
print("Norm=", np.vdot(psi_t, psi_t))

D=
 [[0.96148352 0.        ]
 [0.         2.03851648]]
Eigenvalues= [0.96148352 2.03851648]
Eigenvectors=
 [[-0.98195639+0.j  0.18910752+0.j]
 [ 0.18910752+0.j  0.98195639+0.j]]
psi(t)= [ 0.5357143-0.82263625j -0.1899954-0.01347349j]
Norm= (1.0000000000000004+0j)


## `np.linalg.eigh` vs `scipy.sparse.linalg.eigsh`

Both functions are used to compute:
- eigenvalues
- eigenvectors

```python
eigenvalues, eigenvectors = np.linalg.eigh(H)

eigenvalues, eigenvectors = eigsh(H, k=2)
```

- first returned object → eigenvalues
- second returned object → eigenvectors

### Main difference

| Function | Best for |
|---|---|
| `np.linalg.eigh` | small dense matrices |
| `eigsh` | large sparse matrices |

### Why `eigsh` is important

In quantum computing and numerical simulations, memory is very important.

Large Hamiltonians can become huge, so:

```python
np.linalg.eigh(H)
```

may become:
- slow
- memory expensive

`eigsh` is more efficient because it computes only a few important eigenvalues/eigenvectors, such as:
- ground state energy
- lowest energy states

So `eigsh` is better for large quantum systems.

### Why use `np.linalg.eigh`?

`np.linalg.eigh` is useful for:
- small systems
- dense matrices
- computing all eigenvalues and eigenvectors exactly

### Simple idea

| Function | Best use |
|---|---|
| `eigh` | exact full diagonalization |
| `eigsh` | efficient for large sparse systems |

# Time-dependent Schrödinger Equation (TDSE)

We start from the formal solution

$$
\psi(t)=e^{-iHt}\psi_0
$$

Differentiate with respect to time:

$$
\dot{\psi}(t)
=
\frac{d}{dt}\psi(t)
=
\frac{d}{dt}
\left(
e^{-iHt}\psi_0
\right)
$$

Since $$\psi_0\$$ is constant in time:

$$
\dot{\psi}(t)
=
\frac{d}{dt}
\left(
e^{-iHt}
\right)\psi_0
$$


$$
\dot{\psi}(t)
=
\frac{d}{dt}e^{-iHt}
=
-iHe^{-iHt}
$$


$$
\dot{\psi}(t)
=
-iH
\underbrace{
e^{-iHt}\psi_0
}_{=\psi(t)}
$$

$$
\dot{\psi}(t)
=
-iH\psi(t)
$$

which is exactly the time-dependent Schrödinger Equation.

## Main Idea

Exact diagonalization:

1. Finds eigenvalues and eigenvectors of the Hamiltonian \(H\)
2. Uses them to solve the quantum evolution exactly

So the purpose of exact diagonalization is not only diagonalizing \(H\), but computing the exact dynamics of the system.

## Transition to ODE Solvers

For large systems:

- diagonalization becomes expensive
- memory usage becomes huge
- time-dependent Hamiltonians \(H(t)\) become difficult

So instead of exact evolution, we:
- discretize time into small steps $\Delta t$
- approximate evolution numerically

$$
\psi(t) \rightarrow \psi(t+\Delta t)
$$
means:

- start with the quantum state at time $t$
- compute the next state a small time later at $t+\Delta t$

where:

- $t$ = current time
- $\Delta t$ = small time step

So the ODE solver updates the state step-by-step:

$$
\psi(0)
\rightarrow
\psi(\Delta t)
\rightarrow
\psi(2\Delta t)
\rightarrow
\cdots
$$

instead of solving the whole evolution at once.

This is the motivation for ODE solvers.

### Problems:
* $V$ is **dense** $\Rightarrow V \vec{\psi}_0$ scales as $d^2$ 
  * $+$ memory limitations for large $d$
* For time-dependent problems $H(t)$, this approach is not feasible.

### Solution: Discretize time into steps $\Delta t$

$$\vec{\psi}(t) = \left(e^{-iH\Delta t}\right)^n \vec{\psi}_0 \quad (t = n \Delta t)$$

$$\Rightarrow \text{How to obtain } \vec{\psi}(t+\Delta t) = e^{-iH\Delta t} \vec{\psi}(t) \text{?}$$

* What is the error?
* Is it stable, norm conserving, ...?

## 2) Taylor series propagators

> **Notation Note:**
> * $\Delta t \longleftrightarrow dt$
> * $\vec{\psi} \longleftrightarrow |\psi\rangle$

### - Euler method:

$$
|\psi(t+dt)\rangle
=
e^{-iHdt}|\psi(t)\rangle
$$

The exponential is expanded using the Taylor series:

$$
e^{-iHdt}
=
\frac{(-iHdt)^0}{0!}
+
\frac{(-iHdt)^1}{1!}
+
\frac{(-iHdt)^2}{2!}
+
\cdots
$$


$$
e^{-iHdt}
=
1
-iHdt
+\frac{(-iHdt)^2}{2!}
+\cdots
$$

Since \(dt\) is very small, $$
(dt)^0 > (dt)^1 > (dt)^2 > (dt)^3 > \cdots
$$

higher-order powers become very small, so we keep only the first-order terms:

$$
e^{-iHdt}
\approx
1-iHdt
$$

Therefore,

$$
|\psi(t+dt)\rangle
\approx
(1-iHdt)|\psi(t)\rangle
$$

which is the Euler method approximation.

$$|\psi(t+dt)\rangle = e^{-iHdt}|\psi(t)\rangle = (1 - iHdt)|\psi(t)\rangle + \mathcal{O}(dt^2)$$

* **Error:** $\sim dt^2$ per time step (global error $\sim dt$ for a fixed $t_{\text{end}}$).
* **Stability:** Calculate the norm:

$$\langle\psi(t+dt)|\psi(t+dt)\rangle = \langle\psi(t)|(1 + idtH)(1 - idtH)|\psi(t)\rangle$$

$$= 1 + dt^2 \langle\psi(t)|H^2|\psi(t)\rangle$$

$$\rightarrow \text{\textbf{Norm grows!}}$$

After $n_t$ steps:

$$\|\psi(t)\| = \langle\psi_0|(1 + dt^2 H^2)^{n_t}|\psi_0\rangle \sim e^{t \, dt \, \lambda_{\text{max}}^2}$$

*(where $\lambda_{\text{max}}$ is the largest eigenvalue of $H$)*

$$\Rightarrow \text{\textbf{norm explodes exponentially}} \Rightarrow \text{\textbf{unstable}}$$

$$\Rightarrow \text{need tiny time steps } dt \ll \frac{1}{\lambda_{\text{max}}}$$

---

### - $n$-th order:

$$|\psi(t+dt)\rangle = \left( \sum_{j=0}^{n} \frac{(-idt)^j}{j!} H^j \right) |\psi(t)\rangle + \mathcal{O}(dt^{n+1})$$

* Error per step $\sim dt^{n+1} \rightarrow$ larger steps possible.
* But **still unstable**, not norm-conserving.

---

> **Note:** For a linear ODE $\dot{\psi} = H\psi$, Runge-Kutta methods reduce to the Taylor expansion method.

---

### General stability criterion:

$$|\psi(t+\Delta t)\rangle = A|\psi(t)\rangle : \text{ if } \|A^\dagger A\| > 1 \rightarrow \text{\textbf{unstable}}$$

*(where $\|\cdot\|$ is the 2-norm, i.e., the absolute value of the largest eigenvalue)*

### Stiffness:

$$\left| \frac{\lambda_{\text{max}}}{\lambda_{\text{min}}} \right| \gg 1 \quad \text{problem is "stiff"}$$

* **Physically:** Need small steps to resolve dynamics on the time scale $1/\lambda_{\text{max}}$. Need to integrate to long times to observe dynamics on the time scale $1/\lambda_{\text{min}}$.
  $$\Rightarrow \text{\textbf{Many steps needed.}}$$

---

### Advantage of explicit methods:
* Only $H|\psi\rangle$-multiplication needed.
  $$\rightarrow \text{\textbf{efficient if } } H \text{ \textbf{is sparse!}} \sim d$$

### Stability of the Euler Method

Starting from the Euler update:

$$
|\psi(t+dt)\rangle
=
(1-idtH)|\psi(t)\rangle
$$

Take the Hermitian conjugate:

$$
\langle\psi(t+dt)|
=
\left(
(1-idtH)|\psi(t)\rangle
\right)^\dagger
$$

Using

$$
(AB)^\dagger = B^\dagger A^\dagger
$$

and \(H^\dagger = H\), we obtain

$$
\langle\psi(t+dt)|
=
\langle\psi(t)|(1+idtH)
$$

Now calculate the norm:

$$
\langle\psi(t+dt)|\psi(t+dt)\rangle
=
\langle\psi(t)|
(1+idtH)
(1-idtH)
|\psi(t)\rangle
$$

Expand the product:

$$
=
\langle\psi(t)|
\left(
1-idtH+idtH+dt^2H^2
\right)
|\psi(t)\rangle
$$

The linear terms cancel:

$$
-idtH + idtH = 0
$$

therefore

$$
=
\langle\psi(t)|
\left(
1+dt^2H^2
\right)
|\psi(t)\rangle
$$

which gives

$$
=
1
+
dt^2
\langle\psi(t)|H^2|\psi(t)\rangle
$$

Since

$$
dt^2 > 0
$$

and

$$
\langle\psi(t)|H^2|\psi(t)\rangle \ge 0
$$

the norm becomes larger than \(1\):

$$
\langle\psi(t+dt)|\psi(t+dt)\rangle > 1
$$

Therefore the Euler method is not norm conserving.

The norm grows over time, leading to unstable exponential growth.

So we need another numerical method that preserves the norm and gives stable quantum evolution.

As students and researchers, we look for more stable numerical methods that preserve the norm during quantum time evolution.

The Euler method is not stable because the norm grows with time.

Therefore, we need norm-preserving methods.

One important example is:

## 3) Implicit methods: Crank–Nicolson

The Crank–Nicolson method is more stable and approximately preserves the norm of the wavefunction.

Instead of using only the current time step, it uses information from both:

- the current state \(t\)
- the next state \(t+dt\)

This gives better stability and accuracy for solving the time-dependent Schrödinger Equation.

## 3) Implicit methods: Crank-Nicolson

**Idea:**

$$e^{+i \frac{dt}{2} H} |\psi(t+dt)\rangle = e^{-i \frac{dt}{2} H} |\psi(t)\rangle$$

$$\underbrace{\left(1 + i \frac{dt}{2} H\right)}_{A} \underbrace{|\psi(t+dt)\rangle}_{x} = \underbrace{\left(1 - i \frac{dt}{2} H\right) |\psi(t)\rangle}_{b}$$

* **"Implicit"** $\rightarrow$ To obtain $|\psi(t+dt)\rangle$ we have to solve a linear system $Ax = b$ 
  $$\rightarrow \text{more costly than explicit methods but \textbf{more stable.}}$$
  $$\Rightarrow \text{\textbf{Can do larger time steps!}}$$

---

$$|\psi(t+dt)\rangle = \underbrace{\frac{1 - i \frac{dt}{2} H}{1 + i \frac{dt}{2} H}}_{A} |\psi(t)\rangle$$

$$A^\dagger A = \mathbb{1} \Rightarrow \text{\textbf{norm-conserving}}, \, A \text{ is unitary}$$

$$\text{error } \sim dt^3 \text{ per step}$$

## 3) Implicit methods: Crank–Nicolson

### Idea

Starting from the exact evolution operator:

$$
|\psi(t+dt)\rangle
=
e^{-iHdt}
|\psi(t)\rangle
$$

Split the exponential symmetrically:

$$
e^{+i \frac{dt}{2} H}
|\psi(t+dt)\rangle
=
e^{-i \frac{dt}{2} H}
|\psi(t)\rangle
$$

Now approximate both exponentials to first order:

$$
e^{+i \frac{dt}{2} H}
\approx
1 + i \frac{dt}{2} H
$$

$$
e^{-i \frac{dt}{2} H}
\approx
1 - i \frac{dt}{2} H
$$

Therefore,

$$
\underbrace{
\left(
1 + i \frac{dt}{2} H
\right)
}_{A}
\underbrace{
|\psi(t+dt)\rangle
}_{x}
=
\underbrace{
\left(
1 - i \frac{dt}{2} H
\right)
|\psi(t)\rangle
}_{b}
$$

### Why is it called "implicit"?

To compute the next state $$\|\psi(t+dt)\rangle \$$, we must solve the linear system

$$
Ax=b
$$

So the future state appears on both sides of the equation.

This is more costly than explicit methods like Euler, but much more stable.

Therefore:

$$\Rightarrow
\text{can use larger time steps}$$

### Update formula

Solving for $$\|\psi(t+dt)\rangle \$$:

$$
|\psi(t+dt)\rangle
=
\underbrace{
\frac{
1 - i \frac{dt}{2} H
}{
1 + i \frac{dt}{2} H
}
}_{A}
|\psi(t)\rangle
$$

### Stability and norm conservation

The Crank–Nicolson operator satisfies

$$
A^\dagger A = \mathbb{1}
$$

therefore:

- the norm is conserved
- \(A\) is unitary

so:

$$
\langle\psi(t+dt)|\psi(t+dt)\rangle
=
\langle\psi(t)|\psi(t)\rangle
$$

Unlike Euler, the wavefunction does not grow exponentially.

### Accuracy

The local error per time step is

$$
\sim dt^3
$$

which is much better than the Euler method.

## 4) Krylov subspace propagator

Recall $n$-th order Taylor series:

$$|\psi(t+dt)\rangle = \left( 1 - idtH - \frac{1}{2} dt^2 H^2 \pm \dots \right) |\psi(t)\rangle$$

$$= \text{superpos. of vectors } (-iH)^k |\psi(t)\rangle \quad (k = 0 \dots n)$$
$$\text{with decreasing weight } \frac{dt^k}{k!}$$

$$\rightarrow |\psi(t+dt)\rangle \text{ lies in Krylov space } \mathcal{K}_{n+1}(-iH, |\psi(t)\rangle)$$
$$\text{spanned by vectors } (-iH)^k |\psi(t)\rangle \text{ up to an error } \sim dt^{n+1}$$

---

### Idea: Evolve unitarily in Krylov space.

Recall: $\{q_0, \dots, q_n\}$ are orthonormalized Krylov vectors.

$$Q = \begin{pmatrix} q_0 \dots q_n \end{pmatrix} \updownarrow d \quad \text{rectangular matrix} \quad (\leftrightarrow n)$$

$$h = Q^\dagger H Q \quad \text{Hamiltonian within Krylov space}$$

---

### "Algorithm":
* Project $|\psi(t)\rangle$ into Krylov space
* Evolve with $e^{-ihdt}$. Can be done exactly as $h$ ($n \times n$ tri-diagonal matrix is easily diagonalized)
* Express $|\psi(t+dt)\rangle$ in original basis

---

### Mathematically:

$$|\psi(t+dt)\rangle = Q e^{-ihdt} Q^\dagger |\psi(t)\rangle$$

$$= Q e^{-ihdt} e_1$$

$$\text{(recall that } |\psi(t)\rangle \text{ is the first Krylov-vector } q_0)$$

$$\rightarrow Q^\dagger |\psi(t)\rangle = \begin{pmatrix} 1 \\ 0 \\ \vdots \\ 0 \end{pmatrix} \updownarrow n \equiv e_1 \text{ )}$$

This scheme is obviously unitary (norm-preserving) since $Q e^{-ihdt} Q^\dagger$ is a unitary operator.

**Claim:** This scheme is an order $dt^{n+1}$ propagation just as the $n$-th order Taylor series propagator.

$$Q e^{-ihdt} Q^\dagger = e^{-i Q h Q^\dagger dt} \equiv e^{-i \tilde{H} dt} \overset{!}{=} e^{-i H dt} + \mathcal{O}(dt^{n+1})$$

To prove the last equality, we need to show that:

$$H^k \psi = \tilde{H}^k \psi \quad \text{for } k \le n$$

---

### Proof by induction:

Assuming $H^{k-1} \psi = \tilde{H}^{k-1} \psi$, we have:

$$H^k \psi = \underbrace{Q Q^\dagger H^k \psi}_{H^k \psi \text{ is inside of } \mathcal{K}_{n+1}} = Q Q^\dagger H \underbrace{\tilde{H}^{k-1} \psi}_{\text{ind. assumption}} = Q Q^\dagger H Q h^{k-1} Q^\dagger \psi$$

$$= Q h^k Q^\dagger \psi = \tilde{H}^k \psi$$

$$\begin{pmatrix} q_0 \dots q_n \end{pmatrix} \begin{pmatrix} q_0^\dagger \\ \vdots \\ q_n^\dagger \end{pmatrix} \underbrace{\sum_{k=0}^n c_k q_k}_{\parallel \atop H^k \psi} = \begin{pmatrix} q_0 \dots q_n \end{pmatrix} \begin{pmatrix} c_0 \\ \vdots \\ c_n \end{pmatrix} = \sum c_k q_k = H^k \psi$$

---

### Advantages:
* Error strictly smaller than for $n$-th order Taylor.
* Norm-preserving.
* Only $H\psi$-multiplications (no matrix inversion).

Typically $n \sim 20$ is optimal $\Rightarrow$ large time steps possible.

**"Gold standard"** for propagating SE with sparse $H$.
* Error estimates available.

## 5) Adaptive step-size

**Goal:** In each time step, make the step size $dt$ small enough to meet a given accuracy $\varepsilon$.

* If an error estimate is available (upper bound on error), check and reduce step size by some factor until $\Delta_{\text{err}} \le \varepsilon$. If $\Delta_{\text{err}} < \varepsilon$, keep or increase $dt$.

* **Error estimation by half step:**
  * Calculate $\psi(t+dt)$ with step $dt$
  * Calculate $\tilde{\psi}(t+dt)$ by evolving in 2 half steps $\frac{dt}{2}$
  
$$\rightarrow \text{error estimate } \Delta_{\text{err}} = \|\psi(t+dt) - \tilde{\psi}(t+dt)\|$$